In [2]:
import sys
from pprint import pprint
sys.executable

'C:\\Users\\User\\anaconda3\\envs\\diplom\\python.exe'

In [3]:
import json
import re
import unicodedata
from datetime import datetime
import os

In [4]:
#ТЕКСТ

# ---------- 1. Zero-width и невидимые ----------
ZERO_WIDTH = dict.fromkeys(
    map(ord, '\u200b\u200c\u200d\u2060\ufeff\u180e'), None
)

# ---------- 2. Кавычки, тире, дефисы ----------
PUNCT_MAP = str.maketrans({
    # кавычки -> обычные
    '«': ''', '»': ''', '„': ''', '“': ''', '”': ''', '‟': ''',
    '‘': ''', '’': ''', '‚': ''', '‛': ''', '′': ''', '″': ''',
    # тире/дефисы
    '—': '-', '–': '-', '−': '-', '‒': '-', '―': '-', '‐': '-', '‑': '-',
    # деление/умножение (иногда используются в размерах)
    '×': 'x', '÷': '/',
})

# ---------- 3. Спецсимволы инженерии ----------
# Диаметр, углы, допуски и т.п. — маппим в ASCII-эквиваленты,
# чтобы их можно было распарсить регуляркой.
SPECIAL_MAP = str.maketrans({
    '⌀': 'Ø', '∅': 'Ø', 'Ø': 'Ø', 'ø': 'Ø',   # диаметр
    '°': '°',                                  # градус оставляем (нужен для углов)
    '±': '±',                                  # плюс-минус оставляем
    '№': 'N',                                  # номер
    'µ': 'u', 'μ': 'u',                        # микро (если не важно)
})

# ---------- 4. Неразрывные/узкие пробелы ----------
NBSP_RE = re.compile(r'[\u00A0\u1680\u2000-\u200A\u202F\u205F\u3000]')

# ---------- 5. Управляющие символы ----------
CTRL_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')


def normalize_text(text: str, *, keep_newlines: bool = True) -> str:
    if not isinstance(text, str):
        return ''

    # 1) Unicode NFKC — схлопывает лигатуры, надстрочные цифры, полноширинные формы
    #    ВАЖНО: NFKC превращает '²' в '2', '½' в '1⁄2'. Для чисел это обычно ок.
    text = unicodedata.normalize('NFKC', text)

    # 2) Невидимые символы
    text = text.translate(ZERO_WIDTH)

    # 3) Управляющие символы (кроме \n и \t)
    text = CTRL_RE.sub('', text)

    # 4) Единые переводы строк
    text = text.replace('\r\n', '\n').replace('\r', '\n')

    # 5) NBSP и узкие пробелы -> обычный пробел
    text = NBSP_RE.sub(' ', text)

    # 6) Кавычки/тире/спецсимволы
    text = text.translate(PUNCT_MAP)
    text = text.translate(SPECIAL_MAP)

    # 7) Схлопываем пробелы и табы
    text = re.sub(r'[ \t]+', ' ', text)

    # 8) Убираем пробелы в начале/конце каждой строки
    text = '\n'.join(line.strip() for line in text.split('\n'))

    # '/' НЕ между двумя цифрами
    # (?<!\d) — слева не цифра, (?!\d) — справа не цифра
    SLASH_NOT_BETWEEN_DIGITS = re.compile(r'(?<!\d)(?<!\d\s)\s*/\s*(?!\d)(?!\s\d)')
    SLASH_NOT_BETWEEN_DIGITS.sub(' / ', text)


    # 9) Схлопываем пустые строки
    if keep_newlines:
        text = re.sub(r'\n{2,}', '\n', text)
    else:
        text = re.sub(r'\n+', ' ', text)

    text = re.sub(r'\r?\n', ' ', text)
    text = re.sub(r';', '; ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [5]:
#ДАТЫ

# ---------- Регулярки ----------
# Порядок важен: сначала самые специфичные.

# ISO: 2024-02-01, 2024-2-1  (разделитель строго '-', чтобы не путать с DMY)
RE_ISO = re.compile(r'\b(\d{4})-(\d{1,2})-(\d{1,2})\b')

# Год в начале: 2024.02.01, 2024/02/01
RE_Y4_MD = re.compile(r'\b(\d{4})[./](\d{1,2})[./](\d{1,2})\b')

# Год в конце, 4 цифры: 01.02.2024, 01/02/2024, 1.2.2024
RE_DMY_Y4 = re.compile(r'\b(\d{1,2})[./](\d{1,2})[./](\d{4})\b')

# Год в конце, 2 цифры: 01.02.24, 01/02/24
RE_DMY_Y2 = re.compile(r'\b(\d{1,2})[./](\d{1,2})[./](\d{2})\b')


def _fmt(y: int, m: int, d: int) -> str | None:
    '''Возвращает YYYY-MM-DD или None, если дата невалидна.'''
    try:
        return datetime(y, m, d).strftime('%Y-%m-%d')
    except ValueError:
        return None


def _fmt_2digit_year(y: int) -> int:
    # POSIX: 00-68 -> 2000-2068, 69-99 -> 1969-1999
    return 2000 + y if y <= 68 else 1900 + y


def normalize_dates(text: str, *, day_first: bool = True) -> str:
    '''
    Заменяет все цифровые даты на YYYY-MM-DD.
    day_first: True -> 01/02/2024 = 1 февраля; False -> 1 января (US).
    Невалидные даты (31.02) оставляются как есть.
    '''

    # --- ISO: 2024-02-01 ---
    def _iso(m):
        return _fmt(int(m.group(1)), int(m.group(2)), int(m.group(3))) or m.group(0)

    text = RE_ISO.sub(_iso, text)

    # --- 2024.02.01 / 2024/02/01 ---
    def _y4_md(m):
        return _fmt(int(m.group(1)), int(m.group(2)), int(m.group(3))) or m.group(0)

    text = RE_Y4_MD.sub(_y4_md, text)

    # --- 01.02.2024 / 01/02/2024 ---
    def _dmy_y4(m):
        a, b, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
        # Эвристика: если a > 12 — это точно день, независимо от day_first
        #             если b > 12 — это точно день, значит day_first был неверен
        if a > 12:
            d, mo = a, b
        elif b > 12:
            mo, d = a, b  # b — день
        elif day_first:
            d, mo = a, b
        else:
            mo, d = a, b
        return _fmt(y, mo, d) or m.group(0)

    text = RE_DMY_Y4.sub(_dmy_y4, text)

    # --- 01.02.24 / 01/02/24 ---
    def _dmy_y2(m):
        a, b, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
        y = _fmt_2digit_year(y)
        if a > 12:
            d, mo = a, b
        elif b > 12:
            mo, d = a, b
        elif day_first:
            d, mo = a, b
        else:
            mo, d = a, b
        return _fmt(y, mo, d) or m.group(0)

    text = RE_DMY_Y2.sub(_dmy_y2, text)

    return text

In [6]:
# Единицы измерения

NORMALIZE = {
    # --- длина ---
    'm': 'm', 'm / м': 'm', 'm/м': 'm',
    'м': 'm', 'м/m': 'm',

    'l. m.': 'lin.m', 'l. m. / м.п.': 'lin.m', 'l. m. / п. м.': 'lin.m',
    'l. m./м.п.': 'lin.m', 'l. m./п. м.': 'lin.m', 'l.m. / п.м.': 'lin.m',
    'l.m./м.п.': 'lin.m', 'l.m./п.м.': 'lin.m', 'l.m./пог.м.': 'lin.m',
    'lin. m. / пог. м.': 'lin.m', 'lin.m': 'lin.m', 'lm / мп': 'lin.m',
    'l .m. / пог. м.': 'lin.m', 'm.l. / м.п.': 'lin.m',
    'r.m': 'lin.m', 'p.m.': 'lin.m',
    'м.п': 'lin.m', 'м.п.': 'lin.m', 'п..м.': 'lin.m', 'п.м.': 'lin.m',
    'пог.м': 'lin.m', 'пог.м.': 'lin.m',

    'm 2': 'm2', 'm2': 'm2', 'm2 / м2': 'm2', 'm2 /м2': 'm2', 'm2/м2': 'm2',
    'm 2/ м 2': 'm2', 'м2': 'm2', 'м2/m2': 'm2',
    'шт / м.кв': 'm2', 'шт/м.кв': 'm2',

    'm3': 'm3', 'm3 / м3': 'm3', 'm3/м3': 'm3', 'm 3/ м 3': 'm3', 'м3': 'm3',
    'm3 / м4': 'm3', 'm3 / м5': 'm3',

    # --- масса ---
    'kg': 'kg', 'kgs': 'kg', 'kg / кг': 'kg', 'kg /кг': 'kg', 'kg.': 'kg',
    'kg/кг': 'kg', 'кг': 'kg', 'кг.': 'kg', 'кг/kg': 'kg',

    't': 't', 't.': 't', 't / т': 't', 't / тн': 't', 't / тн.': 't',
    't /тн': 't', 't/т': 't', 't/тн.': 't', 'tn / тн.': 't', 'tn/тн': 't',
    'т': 't', 'т.': 't', 'тн': 't', 'тн / tn': 't',

    # --- объём ---
    'l': 'l', 'l / л': 'l', 'l /л': 'l', 'l/л': 'l',
    'л': 'l', 'л.': 'l', 'liter/литр': 'l',

    # --- штуки ---
    'pc / шт': 'pcs', 'pc/шт': 'pcs', 'pcs': 'pcs', 'pcs / шт': 'pcs',
    'pcs /шт': 'pcs', 'pcs.': 'pcs', 'pcs/шт': 'pcs',
    'ps / шт': 'pcs', 'psc': 'pcs', 'psc / шт': 'pcs', 'psc /шт': 'pcs',
    'psc.': 'pcs', 'psc./шт': 'pcs', 'psc/шт': 'pcs',
    'шт': 'pcs', 'шт.': 'pcs', 'шт/pcs': 'pcs', 'шт/psc': 'pcs',

    # --- пары ---
    'pairs / пары': 'pair', 'pairs/пар': 'pair', 'пар': 'pair',
    'пара': 'pair', 'пары': 'pair',

    # --- наборы ---
    'set / компл': 'set', 'set / комплект': 'set', 'set /комп.': 'set',
    'set /комплект': 'set', 'set/ / компл.': 'set', 'set/комп': 'set',
    'set/компл': 'set', 'компл': 'set', 'компл.': 'set',
    'kit/компл': 'kit',

    # --- упаковки ---
    'pack / упак': 'pack', 'pack.': 'pack', 'pack/упак': 'pack',
    'box / упак': 'box', 'box/упак': 'box',
    'уп': 'pack', 'уп.': 'pack', 'упак': 'pack', 'упак.': 'pack',
    'упак./ packs': 'pack', 'упак/pack': 'pack',
    'упк': 'pack', 'упк.': 'pack',

    # --- рулоны ---
    'rill / рул.': 'roll', 'roll': 'roll', 'roll / рулон': 'roll',
    'rolls': 'roll', 'рулон': 'roll',

    # --- прочее ---
    'cotainer / контейнер': 'container',
    'бухт': 'buht',
    'I.m./м.п.': 'lin.m',
}


# --- предкомпиляция ---
def _build_pattern(mapping: dict):
    # сортируем ключи по длине (по кол-ву символов после схлопывания пробелов),
    # чтобы более длинные варианты матчились раньше
    items = sorted(mapping.items(), key=lambda kv: len(kv[0]), reverse=True)

    alts = []
    for i, (key, _) in enumerate(items):
        # делаем regex: пробелы -> \s+, остальное экранируем
        parts = re.split(r'\s+', key.strip())
        body = r'\s+'.join(re.escape(p) for p in parts)
        alts.append((i, key, body))

    # склеиваем в один regex; (?P<gN>...) чтобы потом понять, что сработало
    combined = '|'.join(f'(?P<g{i}>{body})' for i, _, body in alts)
    # границы: слева/справа — не буква и не цифра.
    # (кириллица + латиница явно, на случай если \w не дружит с юникодом)
    LB = r'(?<![A-Za-zА-Яа-яЁё0-9])'
    RB = r'(?![A-Za-zА-Яа-яЁё0-9])'
    pattern = re.compile(LB + '(?:' + combined + ')' + RB, re.IGNORECASE)

    name_to_canon = {f'g{i}': mapping[key] for i, key, _ in alts}
    return pattern, name_to_canon


_PATTERN, _NAME_TO_CANON = _build_pattern(NORMALIZE)


def normalize_units_in_text(text: str) -> str:
    if not text:
        return text

    def repl(m: re.Match) -> str:
        for name, canon in _NAME_TO_CANON.items():
            if m.group(name) is not None:
                return canon
        return m.group(0)

    return _PATTERN.sub(repl, text)

In [7]:
# def transform(obj):
#     '''изменить текст в словаре'''
#     if isinstance(obj, dict):
#         return {k: transform(v) for k, v in obj.items()}
#     if isinstance(obj, list):
#         if len(obj) == 1:
#             return transform(obj[0])
#         return [transform(v) for v in obj]
#     if isinstance(obj, str):
#         obj = normalize_text(obj)
#         obj = normalize_dates(obj)
#         obj = normalize_units_in_text(obj)
#         return obj      # только строки
#     return obj

In [10]:
def transform(obj, need_keys = (), keys = ()):
    '''изменить текст в словаре'''
    if isinstance(obj, dict):
        return {k: transform(v, need_keys, keys + (k,)) for k, v in obj.items()}
    if isinstance(obj, list):
        if len(obj) == 1:
            return transform(obj[0], need_keys, keys)
        return [transform(v, need_keys, keys) for v in obj]
    if isinstance(obj, str):
        if not need_keys:
            obj = normalize_text(obj, keep_newlines = False)
            obj = normalize_dates(obj)
            obj = normalize_units_in_text(obj)
            return obj      # только строки
        else:
            for need in need_keys:
                if need in keys:
                    obj = normalize_text(obj, keep_newlines = False)
                    obj = normalize_dates(obj)
                    obj = normalize_units_in_text(obj)
                    return obj
    return obj

In [11]:
# Нормализация Y
with open('../Y/data.json', encoding='utf-8') as f:
    data = json.load(f)

data = transform(data)

with open(r'../gold/clean.json', 'w', encoding='utf-8') as file:
    json.dump(data, file, ensure_ascii=False, indent=4)

# pprint(data)

In [12]:
# Нормализация parsed_en
folder_path = r'../parsed_en'
output_folder = r'../gold/parsed_en'
all_items = os.listdir(folder_path)

for item in all_items:
    full_path = os.path.join(folder_path, item)
    if os.path.isfile(full_path) and item.endswith('.json'):
        with open(full_path, encoding='utf-8') as f:
            data = json.load(f)
        
        data = transform(data, need_keys = ('lines', 'text'))
    
        output_file_path = os.path.join(output_folder, item)
        with open(output_file_path, 'w', encoding='utf-8') as file:
            json.dump(data, file, ensure_ascii=False, indent=4)
        

In [13]:
# Тест единицы измерения
with open('../gold/clean.json', encoding='utf-8') as f:
    test = json.load(f)
    
unit = set()
l= []

for t in test.values():
    for m in t['material']:
        if isinstance(m, dict) and 'quantity' in m:
            unit.add(m['quantity']['unit'])
            l.append(m['quantity']['unit'])
       
print(len(l), len(unit))
unit

8613 19


{'1452',
 '429',
 None,
 'box',
 'buht',
 'container',
 'kg',
 'kit',
 'l',
 'lin.m',
 'm',
 'm2',
 'm3',
 'pack',
 'pair',
 'pcs',
 'roll',
 'set',
 't'}

In [22]:
# ТЕСТ дата
tests = [
    # ISO
    '2024-02-01',
    '2024-2-1',
    # год в начале
    '2024.02.01',
    '2024/02/01',
    # год в конце, 4 цифры
    '01.02.2024',
    '01/02/2024',
    '1.2.2024',
    '31.12.2024',
    # короткий год
    '01.02.24',
    '01/02/24',
    '31.12.99',
    # невалидные
    '31.02.2024',
    '32.13.2024',
    '00.01.2024',
    # не даты
    'версия 1.2.3',
    'цена 100.50',
    '100.200.300',
    '10.20.30',       # выглядит как дата, но 20 не месяц -> не заменится
    # смешанное
    'Срок 01.02.2024, начало 2024-03-15, ревизия 2023.11.20',
    # словесные — НЕ должны трогаться
    '1 февраля 2024',
    'February 1, 2024',
]

for t in tests:
    print(f'{t!r:50} -> {normalize_dates(t)!r}')

'2024-02-01'                                       -> '2024-02-01'
'2024-2-1'                                         -> '2024-02-01'
'2024.02.01'                                       -> '2024-02-01'
'2024/02/01'                                       -> '2024-02-01'
'01.02.2024'                                       -> '2024-02-01'
'01/02/2024'                                       -> '2024-02-01'
'1.2.2024'                                         -> '2024-02-01'
'31.12.2024'                                       -> '2024-12-31'
'01.02.24'                                         -> '2024-02-01'
'01/02/24'                                         -> '2024-02-01'
'31.12.99'                                         -> '1999-12-31'
'31.02.2024'                                       -> '31.02.2024'
'32.13.2024'                                       -> '32.13.2024'
'00.01.2024'                                       -> '00.01.2024'
'версия 1.2.3'                                     -> 'версия 

In [23]:
#ТЕСТ текст
tests = [
    # должны получить ' / '
    ('Part/Деталь',            'Part / Деталь'),
    ('Part /Деталь',           'Part / Деталь'),
    ('Part/ Деталь',           'Part / Деталь'),
    ('Part   /    Деталь',     'Part / Деталь'),
    ('Деталь/Part',            'Деталь / Part'),
    ('мм/с',                   'мм / с'),
    ('km/h',                   'km / h'),
    ('слово/слово',            'слово / слово'),
    ('A/Б/C',                  'A / Б / C'),
    ('5/мм',                   '5 / мм'),
    ('кг/5',                   'кг / 5'),
    # НЕ трогаем (число/число)
    ('1/2',                    '1/2'),
    ('01/02/2024',             '01/02/2024'),
    ('2024/01/02',             '2024/01/02'),
    ('1/2/3/4',                '1/2/3/4'),
    # смешанное
    ('срок 01/02/2024 / просрочен', 'срок 01/02/2024 / просрочен'),
    ('1/2 Part/Деталь',        '1/2 Part / Деталь'),
]

for src, expected in tests:
    got = normalize_slash(src)
    mark = '✅' if got == expected else '❌'
    print(f'{mark} {src!r:35} -> {got!r}')

✅ 'Part/Деталь'                       -> 'Part / Деталь'
✅ 'Part /Деталь'                      -> 'Part / Деталь'
✅ 'Part/ Деталь'                      -> 'Part / Деталь'
✅ 'Part   /    Деталь'                -> 'Part / Деталь'
✅ 'Деталь/Part'                       -> 'Деталь / Part'
✅ 'мм/с'                              -> 'мм / с'
✅ 'km/h'                              -> 'km / h'
✅ 'слово/слово'                       -> 'слово / слово'
✅ 'A/Б/C'                             -> 'A / Б / C'
❌ '5/мм'                              -> '5/мм'
❌ 'кг/5'                              -> 'кг/5'
✅ '1/2'                               -> '1/2'
✅ '01/02/2024'                        -> '01/02/2024'
✅ '2024/01/02'                        -> '2024/01/02'
✅ '1/2/3/4'                           -> '1/2/3/4'
✅ 'срок 01/02/2024 / просрочен'       -> 'срок 01/02/2024 / просрочен'
✅ '1/2 Part/Деталь'                   -> '1/2 Part / Деталь'
